In [13]:
from pathlib import Path
import copy, json, os, random, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    vit_b_32, ViT_B_32_Weights,
    swin_t, Swin_T_Weights,
)
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)

## 1. Confirming CUDA is being used

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.14.0+cu130
CUDA runtime: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 2. Reproducibility and project paths

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models" / "architecture_benchmark"
REPORT_DIR = PROJECT_ROOT / "reports" / "architecture_benchmark"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\elesh\Academics\UTS\36127InnovationLab\Development\AutoVision_Model_Development


## 3. Loading the exact baseline split

In [4]:
SPLIT_PATH = DATA_DIR / "interim" / "compcars_resnet50_baseline_split.csv"

df = pd.read_csv(SPLIT_PATH)

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

NUM_CLASSES = df["target"].nunique()

print("Rows:", len(df))
print("Classes:", NUM_CLASSES)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))

Rows: 2000
Classes: 20
Train / Val / Test: 1600 200 200


## 4. Configuring CompCars image location

In [6]:
DEFAULT_COMPCARS = r"G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J\capstone-vehicle-metadata\data\compcars_kaggle\extracted"

LOCAL_COMPCARS_EXTRACTED = Path(
    os.environ.get("COMPCARS_EXTRACTED", DEFAULT_COMPCARS)
)

if not LOCAL_COMPCARS_EXTRACTED.exists():
    raise FileNotFoundError(
        f"CompCars path does not exist: {LOCAL_COMPCARS_EXTRACTED}"
    )

def resolve_local_path(original_path):
    text = str(original_path).replace("\\", "/")
    if "/extracted/" in text:
        relative = text.split("/extracted/", 1)[1]
    elif "/image/" in text:
        relative = "image/" + text.split("/image/", 1)[1]
    else:
        return Path(original_path)
    return LOCAL_COMPCARS_EXTRACTED / Path(*relative.split("/"))

for part in [train_df, val_df, test_df]:
    part["local_file_path"] = part["file_path"].map(resolve_local_path)

sample = pd.concat([train_df, val_df, test_df]).sample(30, random_state=SEED)
if not sample["local_file_path"].map(Path.exists).all():
    raise FileNotFoundError("Some image paths do not resolve.")

print("Image path mapping is good.")

Image path mapping is good.


## 5. Classes, transforms and dataset

In [7]:
target_mapping = (
    df[["target", "class_name"]]
    .drop_duplicates()
    .sort_values("target")
)
class_names = target_mapping["class_name"].tolist()

IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.80, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CompCarsDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(row["local_file_path"]).convert("RGB")
        image = self.transform(image)
        return image, int(row["target"])

## 6. DataLoaders

In [8]:
BATCH_SIZE = 8
NUM_WORKERS = 0

train_loader = DataLoader(
    CompCarsDataset(train_df, train_transform),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    CompCarsDataset(val_df, eval_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    CompCarsDataset(test_df, eval_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print("Batch size:", BATCH_SIZE)

Batch size: 8


## 7. Model builders

In [9]:
def build_model(model_name):
    if model_name == "vit_b32":
        model = vit_b_32(weights=ViT_B_32_Weights.DEFAULT)

        for p in model.parameters():
            p.requires_grad = False

        model.heads.head = nn.Linear(
            model.heads.head.in_features,
            NUM_CLASSES,
        )

        head_params = model.heads.head.parameters()

        def unfreeze_stage2():
            for p in model.encoder.layers[-1].parameters():
                p.requires_grad = True
            for p in model.encoder.ln.parameters():
                p.requires_grad = True

        stage2_lr = 5e-5

    elif model_name == "swin_t":
        model = swin_t(weights=Swin_T_Weights.DEFAULT)

        for p in model.parameters():
            p.requires_grad = False

        model.head = nn.Linear(
            model.head.in_features,
            NUM_CLASSES,
        )

        head_params = model.head.parameters()

        def unfreeze_stage2():
            # Final Swin stage + final norm
            for p in model.features[-1].parameters():
                p.requires_grad = True
            for p in model.norm.parameters():
                p.requires_grad = True

        stage2_lr = 5e-5

    else:
        raise ValueError(model_name)

    return model.to(DEVICE), head_params, unfreeze_stage2, stage2_lr

## 8. Training utilities

In [10]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def topk_correct(logits, targets, k=5):
    k = min(k, logits.shape[1])
    topk = logits.topk(k, dim=1).indices
    return topk.eq(targets.view(-1, 1)).any(dim=1).sum().item()

def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    top1_correct = 0
    top5_correct = 0
    samples = 0

    for images, targets in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
            logits = model(images)
            loss = criterion(logits, targets)

        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        bs = images.size(0)
        total_loss += loss.item() * bs
        top1_correct += (logits.argmax(1) == targets).sum().item()
        top5_correct += topk_correct(logits, targets, 5)
        samples += bs

    return {
        "loss": total_loss / samples,
        "top1": top1_correct / samples,
        "top5": top5_correct / samples,
    }

def train_stage(model, optimizer, epochs, stage_name):
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    history = []
    best_val = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        start = time.perf_counter()

        train_metrics = run_epoch(model, train_loader, optimizer, scaler)
        with torch.inference_mode():
            val_metrics = run_epoch(model, val_loader)

        elapsed = time.perf_counter() - start

        row = {
            "stage": stage_name,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_top1": train_metrics["top1"],
            "val_loss": val_metrics["loss"],
            "val_top1": val_metrics["top1"],
            "val_top5": val_metrics["top5"],
            "seconds": elapsed,
        }
        history.append(row)

        print(
            f"[{stage_name}] {epoch}/{epochs} | "
            f"train top1={row['train_top1']:.3f} | "
            f"val top1={row['val_top1']:.3f} "
            f"top5={row['val_top5']:.3f} | "
            f"{elapsed:.1f}s"
        )

        if row["val_top1"] > best_val:
            best_val = row["val_top1"]
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return history, best_val, best_state

## 9. Evaluation

In [11]:
def evaluate(model):
    model.eval()
    labels_all = []
    preds_all = []
    probs_all = []

    with torch.inference_mode():
        for images, targets in tqdm(test_loader, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)

            labels_all.extend(targets.numpy())
            preds_all.extend(probs.argmax(1).cpu().numpy())
            probs_all.append(probs.cpu())

    y_true = np.asarray(labels_all)
    y_pred = np.asarray(preds_all)
    probs = torch.cat(probs_all)

    top1 = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    top5_idx = probs.topk(min(5, NUM_CLASSES), dim=1).indices.numpy()
    top5 = np.mean([
        true in candidates
        for true, candidates in zip(y_true, top5_idx)
    ])

    report = pd.DataFrame(
        classification_report(
            y_true, y_pred,
            labels=list(range(NUM_CLASSES)),
            target_names=class_names,
            output_dict=True,
            zero_division=0,
        )
    ).T

    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

    return top1, top5, macro_f1, report, cm

## 10. Running ViT-B/32 and Swin-T

In [14]:
STAGE1_EPOCHS = 3
STAGE2_EPOCHS = 3

new_results = []

for model_name in ["vit_b32", "swin_t"]:
    print("\n" + "=" * 70)
    print("TRAINING:", model_name)
    print("=" * 70)

    model, head_params, unfreeze_stage2, stage2_lr = build_model(model_name)

    optimizer = torch.optim.AdamW(
        head_params, lr=1e-3, weight_decay=1e-4
    )
    hist1, val1, state1 = train_stage(
        model, optimizer, STAGE1_EPOCHS, "head"
    )

    unfreeze_stage2()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=stage2_lr,
        weight_decay=1e-4,
    )
    hist2, val2, state2 = train_stage(
        model, optimizer, STAGE2_EPOCHS, "fine_tune"
    )

    if val1 > val2:
        model.load_state_dict(state1)
        selected_stage = "head"
        best_val = val1
    else:
        model.load_state_dict(state2)
        selected_stage = "fine_tune"
        best_val = val2

    top1, top5, macro_f1, report, cm = evaluate(model)

    model_report_dir = REPORT_DIR / model_name
    model_report_dir.mkdir(parents=True, exist_ok=True)

    pd.DataFrame(hist1 + hist2).to_csv(
        model_report_dir / "training_history.csv",
        index=False,
    )
    report.to_csv(model_report_dir / "classification_report.csv")

    fig, ax = plt.subplots(figsize=(16, 16))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax, xticks_rotation=90, colorbar=False, values_format="d"
    )
    ax.set_title(f"{model_name} — Test Confusion Matrix")
    plt.tight_layout()
    plt.savefig(model_report_dir / "confusion_matrix.png", dpi=160)
    plt.close()

    model_path = MODEL_DIR / f"{model_name}_compcars_baseline.pt"
    torch.save(
        {
            "architecture": model_name,
            "state_dict": model.state_dict(),
            "classes": class_names,
            "num_classes": NUM_CLASSES,
            "selected_stage": selected_stage,
            "validation_top1": float(best_val),
            "test_top1": float(top1),
            "test_top5": float(top5),
            "test_macro_f1": float(macro_f1),
        },
        model_path,
    )

    metrics = {
        "Model": model_name,
        "Validation Top-1": best_val,
        "Test Top-1": top1,
        "Test Top-5": top5,
        "Macro F1": macro_f1,
        "GPU": GPU_NAME,
    }
    new_results.append(metrics)

    with (model_report_dir / "metrics.json").open("w") as f:
        json.dump(metrics, f, indent=2)

    print(metrics)

    del model
    torch.cuda.empty_cache()


TRAINING: vit_b32


[head] 1/3 | train top1=0.260 | val top1=0.410 top5=0.725 | 78.3s


[head] 2/3 | train top1=0.538 | val top1=0.445 top5=0.835 | 59.3s


[head] 3/3 | train top1=0.665 | val top1=0.485 top5=0.875 | 55.1s


[fine_tune] 1/3 | train top1=0.756 | val top1=0.600 top5=0.945 | 54.5s


...

[head] 3/3 | train top1=0.519 | val top1=0.490 top5=0.880 | 56.2s


[fine_tune] 1/3 | train top1=0.680 | val top1=0.680 top5=0.955 | 59.6s


[fine_tune] 2/3 | train top1=0.822 | val top1=0.740 top5=0.975 | 59.5s


[fine_tune] 3/3 | train top1=0.887 | val top1=0.800 top5=0.980 | 59.9s


{'Model': 'swin_t', 'Validation Top-1': 0.8, 'Test Top-1': 0.79, 'Test Top-5': np.float64(0.98), 'Macro F1': 0.7935262796485706, 'GPU': 'NVIDIA GeForce RTX 5060 Laptop GPU'}


## 11. Comparing all four architectures

In [15]:
comparison_rows = []

existing = [
    ("ResNet50", PROJECT_ROOT / "reports" / "baseline_resnet50" / "metrics.json"),
    ("ViT-B/16", PROJECT_ROOT / "reports" / "baseline_vit_b16" / "metrics.json"),
]

for display_name, path in existing:
    if path.exists():
        with path.open() as f:
            m = json.load(f)
        comparison_rows.append({
            "Model": display_name,
            "Test Top-1": m.get("test_top1_accuracy"),
            "Test Top-5": m.get("test_top5_accuracy"),
            "Macro F1": m.get("test_macro_f1"),
        })

for row in new_results:
    comparison_rows.append({
        "Model": row["Model"],
        "Test Top-1": row["Test Top-1"],
        "Test Top-5": row["Test Top-5"],
        "Macro F1": row["Macro F1"],
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.sort_values("Test Top-1", ascending=False))

comparison_df.to_csv(
    REPORT_DIR / "architecture_comparison.csv",
    index=False,
)

,Model,Test Top-1,Test Top-5,Macro F1
0,ResNet50,0.810,0.995,0.810771
3,swin_t,0.790,0.980,0.793526
1,ViT-B/16,0.740,0.990,0.741829
2,vit_b32,0.695,0.970,0.689597
